# Reinforcement Learning

## Q-Learning

Q-Learning reprezintă un algoritm de învățare de tip RL care antrenează un agent să atribuie pentru fiecare acțiune luată într-un anumit mediu o anumită valoare,
ținând cont de starea curentă, dar nefiind necesar un model al mediului (nu trebuie să se știe tot mediu, ci agentul doar să acționeze cu mediul, acesta recompensându-l
pe baza acțiunii luate)



### Mountain Car Continuous

#### Introducere

Mountain Car prezintă problema unei mașini care trebuie să ajungă la un punct final în vârful unui munte.

Mașina prezintă poziția pe axa X și axa Y, deși mediul transmite ca informații doar poziția de pe X (intern verifică folosind poziția de pe Y dacă s-a finalizat un episod, verificând dacă înălțimea la care se află mașina e mai mare sau egală cu valoarea 0.45, vârful fiind cel mai înalt din mediu)

De asemenea, mediul mai oferă și viteza mașinii pe axa X (componenta pe axa X din vectorul viteză atribuit mașinii) pe lângă poziția pe X.





#### Problema

Se cere să se antreneze un agent de tip RL astfel încât acesta să poată urca dealul.

#### Observații

- Spațiul de observare este reprezentat de un spațiu continuu (un Box), care prezintă 2 dimensiuni (viteza și poziția mașinii sunt observate de mediu)

- Spațiul de acțiune este reprezentat tot continuu, ca un scalar (forța aplicată direcției mașinii pentru a o face să meargă în stânga sau în dreapta)

- Mașina începe un episod la o poziție x în intervalul [-0.6, -0.4]

- Mașina este penalizată pentru acțiunii care sunt cu magnitudine mari la fiecare pas (adică o viteză mare cu pătratul forței aplicate înmulțit cu 0.1)

- Mașina primește o recompensă mare la final (+100) când atinge obiectivul.



#### Probleme în învățarea agentului

Prima problemă cea mai mare întrucât se folosește algoritmul de tip tabular Q-Learning este că trebuie să discretizăm spațiul acțiuniilor și al observațiilor (stărilor).

Pentru a realiza acest lucru, am ales spațiul pentru spațiul observațiilor, pentru
viteză și poziție să împart intervalul real în 15 subintervale egale (prin binning)

La fel s-a procedat și pentru spațiul acțiuniilor, folosindu-se 5 bin-uri

Se remarcă că, dacă stăm să comparăm cu Mountain Car varianta discretă pentru spațiul de acțiuni (-1, 0 sau 1), acest mediu conduce la o actualizare mai greoaie a tabele de Q-Learning, întrucât numărul mai mare de diferite forțe direcționale conduce la o căutare mai laborioasă, la o adică la un timp mai lung de antrenare a agentului în comparație cu varianta discretă. Pentru a spori șansele agentului de a ajunge mai repede spre o soluție, s-a folosit o metodă de alterare a recompensei, unde la recompensa primită de agent din partea mediului, s-a adăugat bonus-uri pentru cât de aproape este de partea dreaptă pe axa x (întrucât recompensa este pe axa X) și pentru o viteză ridicată (întrucât avem nevoie de o viteză destul de mare cât să urcăm dealul)



#### Implementare

S-a realizat cu ajutorul librăriei Gymnasium (care ne oferă medii pentru agenți și posibilitatea
de a înregistra anumite episoade pe baza anumitor reguli)


Pentru a putea avea o fază de explorare și exploatare, s-a implementat în cadrul algoritmului
politica de tip Epsilon-Greedy, dar care să prezinte și un decay pe parcursul treceri prin mai multe episoade (astfel agentul începe inițial într-o fază în care explorează cât mai mult mediul ca să poată popula tabela Q, iar pe la finalul învățării epsilon scade așa mult că adesea ia cea mai „bună” acțiune pentru o anumită stare(scris în ghilimele întrucât acea acțiune poate să nu fie optimă, din această cauza chiar și în faza aceasta am ales să existe un prag minim pentru epsilon, astfel încât să existe măcar intenția rară de a alege la întâmplare ceva nou, pentru a scăpa de minime locale)

De asemenea, pentru a putea vizualiza cum evoluează agentul, s-a realizat o metodă care
returnează și pentru un număr N de episoade, care este media recompenselor primite per episod. Astfel, putem analiza și spune dacă în medie agentul reușeste după un număr de episoade să învețe ceva sau nu.

#### Aplicația

##### Faza de antrenare

In [ ]:
import math
import numpy as np
import  gymnasium as gym

from q_learn.q_learning.discretizer import UniformStateDiscretizer, UniformActionDecoder
from q_learn.q_learning.policy import EpsilonGreedyPolicy
from q_learn.q_learning.q_learning import QLearning

from matplotlib import pyplot as plt

# Defining the environment
train_env = gym.make('MountainCarContinuous-v0', render_mode=None)

# Mountain Car observation space is a 2D-vector, being suitable for tabular Q-learning (not many dimensions -> not many entries in the table)

low = [-1.2, -0.07]
high = [0.6, 0.07]
bins = [15, 15]

number_of_states = int(np.prod(bins))

state_encoder = UniformStateDiscretizer(low=low, high=high, bins=bins)

# The action space is continuous

low = [-1]
high = [1]
bins = [5]

number_of_actions = int(np.prod(bins))

action_decoder = UniformActionDecoder(lows=low, highs=high, bins=bins, strategy='center')

# Setup

episodes = 2000

epsilon_start = 1.0
decay_rate = 0.8

epsilon_decay_function = lambda ep, eps: epsilon_start * math.exp((-2*ep * decay_rate) / episodes)

policy = EpsilonGreedyPolicy(epsilon=epsilon_start, epsilon_decay_function=epsilon_decay_function, epsilon_min=0)

agent = QLearning(env=train_env,
                  explorer=policy,
                  state_encoder=state_encoder,
                  action_decoder=action_decoder,
                  learning_rate=0.1,
                  discount_factor=0.8,
                  number_of_actions=number_of_actions,
                  number_of_states=number_of_states
                  )

averaging_step = 100

def reward_func(reward: float, env: gym.Env, state: gym.core.ObsType, next_state: gym.core.ObsType, action: gym.core.ActType) -> float:

    position, velocity = next_state

    # we want to encourage moving right
    reward += (position + 1.2) * 0.05 # (+ 1.2 to go from [-1.2, 0.6] to [0, 1.8])

    # encourage high velocity as well (since high velocity will help in reaching the goal without needing constant acceleration)
    reward += abs(velocity) * 0.2

    return reward

average_rewards, q_table = agent.run(number_of_episodes=episodes, averaging_step=averaging_step, reward_hook_func=reward_func)


# Plotting average rewards
plt.plot((averaging_step * np.arange(len(average_rewards)) + 1), average_rewards)
plt.title('Q-Learning Mountain Car Average Rewards')
plt.xlabel('Episode')
plt.ylabel('Average Reward')
plt.show()

train_env.close()

np.save('table.npy', q_table)


În codul de mai sus se instanțiază un nou mediu folosind Gymnasium `train_env`, după care se definesc limitele spațiului de observare `low`, `high`, și
câte bin-uri să avem în cadrul procesului de binning `bins`

Apoi, se realizează codificarea stărilor cu ajutorul unei clase de utilitate `UniformStateDiscretizer`, care are ca scop codificarea stărilor continue
în stări discrete.

S-a procedat la fel și pentru spațiul acțiunilor care e continuu.

S-a setat numărul de episoade (2000), după care pentru politica de alegerea a acțiuni (epsilon-greedy), s-a folosit o funcție de decădere a valorii
epsilon (astfel încât epsilon să înceapă cu o valoare mare să încurajeze explorarea, scăzând pe parcursul episoadelor ducând la exploatarea tabelei Q.
(De remarcat că epsilon scade până la o anumită valoare de prag, întrucât deși tabela Q e populată, nu vrem să avem ghinionul de a avea optime locale, așa că mai permitem și alegerea random în situații puține a unei acțiuni în cadrul unei stări.)

Pentru a putea analiza evoluția agentului, la fiecare 100 de episoade se ia media recompenselor obținute, ca ulterior să se afișeze.

Am definit o funcție de recompense, care să sporească șansele agentului de a reuși (cu cât se apropie de dreapta e mai bine, la fel și dacă are viteză mare,
întrucât înseamnă că are șanse mai mari să reușească). Aceste mini-recompense sunt adăugate la recompensa primită de mediu.

S-a rulat algoritmul care returnează valorile medii de recompense și tabela Q finală.

Am afișat evoluția mediei recompenselor în cadrul episoadelor pentru a vedea dacă agentul a învățat sau nu ceva

În cadrul algoritmului am ales și learning_rate=0.1, pentru a permite agentului să învețe, dar să nu piardă din vedere potențiale acțiunii care
pot fi exploatate. De asemenea, discount_factor=0.8 întrucât agentul depinde de recompense din viitorul îndepărtat (întrucât recompensele primite de la mediu
sunt aproape mereu negative și pozitive doar la atingerea obiectivului)

S-a salvat tabela și astfel s-a terminat faza de „antrenament” (în care lăsăm agentul să exploreze astfel încât să putem pregăti tabela Q, ca ulterior să o refolosim)



##### Faza de testare

In [ ]:
arr = np.load('table.npy')

test_env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')

greedy_policy = EpsilonGreedyPolicy(epsilon=0.05)

eval_agent = QLearning(env=test_env,
                       explorer=greedy_policy,
                       state_encoder=state_encoder,
                       action_decoder=action_decoder,
                       number_of_actions=number_of_actions,
                       number_of_states=number_of_states,
                       shall_record=True,
                       episode_record_policy=lambda _: True)

eval_agent.q_table = arr

rwds, _= eval_agent.run(number_of_episodes=5)

print(rwds)

test_env.close()

În cadrul fazei de testare, s-a refăcut mediul, de data asta permițând și înregistrarea episoadelor (pentru a putea vedea și vizual cum se descurcă agentul)
Se observă că refolosim tabela Q deja populată anterior și rulăm iar algoritmul pentru 5 iterații, folosind aceași politică epsilon-greedy doar că de data asta
având un epsilon fix, setat la 0.05 (astfel, agentul în 95% din cazuri folosește cea mai bună acțiune pentru a exploata tabela Q, dar permite și 5% cazuri în care să aleagă random pentru a avea și diversitatea soluțiilor și a evita optime locale.


##### Concluzii



###### Generale

Se poate observa că agentul reușește după multe episoade să își actualizeze tabela Q pentru a învăța cele mai bune acțiuni într-o anumită stare pentru a-și maximiza recompensa. Totuși, întrucât sistemul de recompense pentru Mountain Car implică adesea pedepse și doar la final recompensă mare, a trebuit să se realize un mecanism de „hooking” în sistemul de recompense, prin care să sporim recompensele puțin mai mult primite de agent, întrucât altfel agentul ar fi trebuit
să aibă parte mai mult de noroc pentru a ajunge la soluția problemei.


Algoritmul Tabular Q-Learning în cazul de față se pretează datorită numărului redus de dimensiuni în cadrul spațiului de acțiuni și observații. Prin binning, s-a
putut realiza maparea fiecărei acțiuni/stări la un indice în tabela Q, iar prin procesul invers (trecerea de la un indice la valoarea din bin), s-a putut afla
valoarea acțiunii din tabelă și folosirea ei în interacțiunea cu mediul.


Comparativ cu acest mediu, LunarLander nu ar fi putut rezolva cu Tabular Q-Learning, ci mai degrabă cu DQN (_Deep Q-Network_), întrucât valoarea acțiunii luate în
starea curentă nu putea fi stocată nicăieri, întrucât tabelul ar fi fost extrem de mare, consumând o grămadă de memorie, folosindu-se DQN cu aproximarea valorii folosind rețele neuronale.


Este nevoie de un număr mare de episoade pentru ca agentul să învețe. Acest lucru e datorat faptului că agentul are o grămadă de acțiunii și stării în care poate fi (fapt influențat și de dimensiunea bin-urilor), tabela Q crescând odată cu dimensiunea numărului de stări și acțiuni.



###### Evaluarea agentului

Se observă că agentul, în decursul numărului de episoade, ajunge să învețe că trebuie să se îndrepte cât mai mult spre dreaptă (datorită modificări sistemului de recompense). De asemenea, acesta învață că are nevoie de viteză mare (ca un fel de pendul, pentru a avea viteza necesară de a urca dealul). Evident, acest comportament a fost indus mai mult de sistemul modificat de recompense, pentru a spori șansa agentului să ajungă la target. Altfel, agentul ar fi ajuns cam la aceleași concluzii, dar ar fi durat mai mult, întrucât ar fi fost recompensat doar la final.

### Cod sursă

#### q_learning.py

Fișierul principal în care se implementează algoritmul de tabular Q-Learning

````python
import os
from collections.abc import Callable

import cv2
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from pygame.math import clamp

from q_learn.q_learning.policy import Policy
import numpy as np

from q_learn.q_learning.qtypes import RewardHook


class QLearning:

    def __init__(self, env: gym.Env, explorer: Policy,
                 number_of_states: int,
                 number_of_actions: int,
                 learning_rate: float = 0.3,
                 discount_factor: float = 0.7,
                 shall_record: bool = False,
                 episode_record_policy: Callable[[int], bool] = lambda ep: False,
                 video_folder: str = "videos",
                 name_prefix: str = "test",
                 fps: int = 25,
                 successful_episode_record_policy: Callable[[list[list], list[float]], bool] = lambda frames, rewards: False,
                 state_encoder=None, # used for continuous observation spaces (coming from env)
                 action_decoder=None, # used for continuous action spaces (since q-table actions are indices, we need a decoder to get back in continuous space)
                 ):
        """
        Construct a Q-Learning object meant to run the tabular Q-learning algorithm.
        :param env: The environment to run the Q-learning on, provided by Gymnasium
        :param number_of_states: The number of states to use in the Q-learning table
        :param number_of_actions: The number of actions to use in the Q-learning table
        :param state_encoder: The state encoder to use in the Q-learning table (for continuous states coming from the environment)
        :param action_decoder: The action decoder to use in the Q-learning table (for an index to get the real action value)
        :param explorer: The policy to use for exploration
        :param learning_rate: The learning rate
        :param discount_factor: The discount factor (Gamma)
        :param shall_record: If set to `True`, it enables episode recording, otherwise it disables it
        :param episode_record_policy: Policy to use for episode recording based on the index of the episode
        :param video_folder: The folder where to save videos to
        :param name_prefix: The prefix a video saved has
        :param fps: The frames per second
        :param successful_episode_record_policy: Function that upon returning `True` permits saving the episode
        based on its rewards or frames. *This policy differs based on the environment reward system*.
        """
        self.env = env
        self.policy = explorer
        self.learning_rate = clamp(learning_rate, 0, 1)
        self.discount_factor = clamp(discount_factor, 0, 1)
        self.fps = fps

        self.shall_record = shall_record

        self.successful_trigger = successful_episode_record_policy

        if number_of_states is None:
            raise ValueError("number_of_states cannot be None")

        if number_of_actions is None:
            raise ValueError("number_of_actions cannot be None")

        n_states = number_of_states
        n_actions = number_of_actions

        # encoder
        # since gym gives us the state, if it is continuous, then we need to encode it (to discrete values)
        self.state_encoder = state_encoder

        # decoder
        # since q-table gives us the action, we need to decode it from discrete to it's supposed continuous value
        self.action_decoder = action_decoder


        self.number_of_states = n_states
        self.number_of_actions = n_actions

        self.q_table = np.zeros((n_states, n_actions))

        self.video_folder = video_folder
        self.name_prefix = name_prefix

        self.episode_trigger = episode_record_policy

        if shall_record:
            self.env = RecordVideo(self.env, video_folder=video_folder, name_prefix=name_prefix, episode_trigger=episode_record_policy,
                                   fps=fps)

        pass

    def reset_table(self):
        self.q_table = np.zeros((self.number_of_states, self.number_of_actions))


    def encode_state(self, state):
        return self.state_encoder(state) if self.state_encoder else state

    def decode_action(self, action_idx):
        return self.action_decoder([action_idx]) if self.action_decoder else action_idx



    def run(self, number_of_episodes: int = 5,
            averaging_step: int = 100,
            reward_hook_func: RewardHook = None):

        """
        Run Q-Learning episodes
        :param number_of_episodes: The number of episodes to run
        :param averaging_step: The episode multiple at which to average the reward
        :param reward_hook_func: Function to alter reward result that is given by the Gymnasium environment
        :return: rewards, steps, Q-table
        """

        # Returned data
        rewards = []


        average_rewards = []


        for episode_num in range(number_of_episodes):

            raw_state, info = self.env.reset()

            state = self.encode_state(raw_state)

            assert 0 <= state < self.number_of_states

            step = 0
            total_rewards = 0

            episode_frames = []
            episode_rewards = []

            # initial frame

            if self.shall_record:
                frame = self.env.render()
                episode_frames.append(frame)

            success = None
            episode_over = False
            while not episode_over:

                action_idx = self.policy.select_action(state, self.q_table, self.env, episode_num)

                # decode action from q-table
                action = self.decode_action(action_idx)

                raw_next_state, reward, terminated, truncated, info = self.env.step(action)

                if reward_hook_func:
                   reward = reward_hook_func(reward=reward,
                                             env=self.env,
                                             state=raw_state,
                                             next_state=raw_next_state,
                                             action=action)

                # encode for q-table storage
                next_state = self.encode_state(raw_next_state)

                assert 0 <= next_state < self.number_of_states


                # frame after step
                if self.shall_record:
                    frame_after = self.env.render()
                    episode_frames.append(frame_after)

                episode_rewards.append(reward)

                delta = (
                        reward
                        + self.discount_factor * np.max(self.q_table[next_state])
                        - self.q_table[state, action_idx]
                )

                self.q_table[state, action_idx] = self.q_table[state, action_idx] + self.learning_rate * delta

                state = next_state

                step += 1
                total_rewards += reward

                episode_over = truncated or terminated
                success = terminated


            if self.shall_record:
                if self.successful_trigger:
                    if self.successful_trigger(episode_frames, episode_rewards) or success is True:
                        self.save_episode_video(episode_frames, episode_num)


            rewards.append(total_rewards)

            if (episode_num + 1) % averaging_step == 0:
                average_rewards.append(np.mean(rewards))
                rewards = []


        return average_rewards, self.q_table

    def save_episode_video(self, frames: list[list], episode_num: int):
        os.makedirs(self.video_folder, exist_ok=True)
        height, width, _ = frames[0].shape
        filename = os.path.join(self.video_folder, f"{self.name_prefix}-episode-{episode_num}.mp4")
        out = cv2.VideoWriter(filename, cv2.VideoWriter_fourcc(*'mp4v'), self.fps, (width, height))
        for f in frames:
            f_bgr = cv2.cvtColor(f, cv2.COLOR_RGB2BGR)
            out.write(f_bgr)
        out.release()
````



#### policy.py

Aici s-au definit diverselor metode de selecție a acțiunii de către agent.

````python
from abc import ABC, abstractmethod
from typing import Callable

import numpy as np
import gymnasium as gym


class Policy(ABC):
    """Base class for Q-Learning policy"""

    @abstractmethod
    def select_action(self, state, q_table, env: gym.Env, episode: int = 0):
        """
        Returns an action given the current `state`, using the tabular Q-Learning table `q_table`,
        using an environment `env` supplied by `Gymnasium`
        :param state: The current state
        :param q_table: The Q-Learning table
        :param env: The environment that comes from Gymnasium
        :param episode: The episode at which the action is taken
        :return: An action that can be taken in the environment
        """
        pass


class RandomPolicy(Policy):
    def select_action(self, state, q_table, env, episode: int = 0):
        return np.random.randint(q_table.shape[1])


class GreedyPolicy(Policy):
    def select_action(self, state, q_table, env, episode: int = 0):
        return np.argmax(q_table[state])


class EpsilonGreedyPolicy(Policy):

    def __init__(self, epsilon=0.2, epsilon_decay_function: Callable[[int, float], float]=None, epsilon_min: float=0.01):
        self.epsilon = epsilon
        self.decay_function = epsilon_decay_function if epsilon_decay_function else lambda ep, eps: epsilon
        self.epsilon_min = epsilon_min

    def select_action(self, state, q_table, env, episode: int = 0):

        self.epsilon = max(self.epsilon_min, self.decay_function(episode, self.epsilon))

        if np.random.uniform() < self.epsilon:
            action = np.random.randint(q_table.shape[1])
        else:
            action = np.argmax(q_table[state])

        return action
````



#### discretizer.py

Aici se regăsesc metode de utilitate în cadrul procesului de binning.

````python
import numpy as np
import gymnasium as gym
from numpy import dtype


class UniformStateDiscretizer:
    def __init__(self, low, high, bins):
        self.low = np.array(low)
        self.high = np.array(high)
        self.bins = np.array(bins)

    def __call__(self, obs: gym.core.ObsType):
        obs = np.clip(obs, self.low, self.high) # clipping extreme values

        ratios = (obs - self.low) / (self.high - self.low) # from uniform quantization to get the bin (bucket) index

        indices = np.floor(ratios * self.bins).astype(int) # modified formula to truncate
        indices = np.clip(indices, 0, self.bins - 1) # floor the ratios to get integers, then clamp values outside (0, bin_number)

        # we have 8 indices now, we need one unique value from it
        # if we have something like:
        # bins = [4, 3, 2]
        # indices = [2, 1, 0]

        # then, bins[0] tells us that the first feature has 4 possible values, bins[1] tells us 3 and so on
        # this would mean, in total, a number of values equal to 4*3*2 = 24 total possibilities

        # these will be actions/states to be encoded, so we need to get a unique index
        # the easiest way is to generate something like a 1-D array from the bins, then return the index of the indices element, which is unique

        # this is where ravel_multi_index does the heavy lifting, by doing exactly that
        # it flattens a multidimensional grid, then returns the index for the element in that grid

        # for performance reason, it uses a mathematical formula directly to generate the index
        return np.ravel_multi_index(indices, self.bins)


class UniformActionDecoder:
    """
    Decodes discrete indices to continuous actions for N-dimensional action spaces.
    """

    def __init__(self, lows, highs, bins, strategy="center"):
        self.lows = np.array(lows, dtype=float)
        self.highs = np.array(highs, dtype=float)
        self.bins = np.array(bins, dtype=int)
        self.strategy = strategy

        self.widths = (self.highs - self.lows) / self.bins
        self.dim = len(self.lows)

        if len(self.highs) != self.dim or len(self.bins) != self.dim:
            raise ValueError("lows, highs, bins must all have the same length")

    def __call__(self, indices):
        """
        :param indices: An array of discrete indices. (have to be integers)
        Returns: np.array of decoded continuous action values
        """
        indices = np.array(indices, dtype=int)
        if len(indices) != self.dim:
            raise ValueError(f"indices length {len(indices)} != number of dimensions {self.dim}")

        actions = np.zeros(self.dim)


        for i in range(self.dim):
            idx = np.clip(indices[i], 0, self.bins[i] - 1)
            if self.strategy == "left":
                actions[i] = self.lows[i] + idx * self.widths[i]
            elif self.strategy == "right":
                actions[i] = self.lows[i] + (idx + 1) * self.widths[i]
            elif self.strategy == "center":
                actions[i] = self.lows[i] + (idx + 0.5) * self.widths[i]
            else:
                raise ValueError(f"Unknown strategy {self.strategy}")

        return actions
````

#### qtypes.py

Fișier pentru a defini tipuri pentru o mai bună înțelegere a API-ului dezvoltat.

````python
from typing import Protocol
import gymnasium as gym

class RewardHook(Protocol):
    def __call__(self,
                 reward: float,
                 env: gym.Env,
                 state: gym.core.ObsType,
                 next_state: gym.core.ObsType,
                 action: gym.core.ActType) -> float:
        pass
````